<a href="https://colab.research.google.com/github/IsaganiJulian/Guardian-Recruit-Fraud-Detection/blob/feature%2Foutlier-detection-kusuma/notebooks/sandbox_outlier_kusuma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving emscad_dataset.csv to emscad_dataset.csv


In [2]:
import pandas as pd

df = pd.read_csv("emscad_dataset.csv")
df.head()

,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset
0,Marketing Intern,"US, NY, New York",Marketing,NaN,"<h3>We're Food52, and we've created a groundbr...","<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,f,t,f,Other,Internship,NaN,NaN,Marketing,f,f
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"<h3>90 Seconds, the worlds Cloud Video Product...",<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,<h3><b>What you will get from us</b></h3>\r\n<...,f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,f,f
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,<h3></h3>\r\n<p>Valor Services provides Workfo...,"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,f,t,f,NaN,NaN,NaN,NaN,NaN,f,f
3,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,<p>Our passion for improving quality of life t...,<p><b>THE COMPANY: ESRI – Environmental System...,<ul>\r\n<li>\r\n<b>EDUCATION: </b>Bachelor’s o...,<p>Our culture is anything but corporate—we ha...,f,t,f,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,f,f
4,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,<p>SpotSource Solutions LLC is a Global Human ...,<p><b>JOB TITLE:</b> Itemization Review Manage...,<p><b>QUALIFICATIONS:</b></p>\r\n<ul>\r\n<li>R...,<p>Full Benefits Offered</p>,f,t,t,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,f,f


In [3]:
df.shape

(17880, 18)

In [4]:
df.columns

Index(['title', 'location', 'department', 'salary_range', 'company_profile',
       'description', 'requirements', 'benefits', 'telecommuting',
       'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent', 'in_balanced_dataset'],
      dtype='object')

In [5]:
metadata_cols = [
    'location', 'department', 'salary_range',
    'telecommuting', 'has_company_logo', 'has_questions',
    'employment_type', 'required_experience',
    'required_education', 'industry', 'function'
]

meta_df = df[metadata_cols].copy()

meta_df.head()

,location,department,salary_range,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function
0,"US, NY, New York",Marketing,NaN,f,t,f,Other,Internship,NaN,NaN,Marketing
1,"NZ, , Auckland",Success,NaN,f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service
2,"US, IA, Wever",NaN,NaN,f,t,f,NaN,NaN,NaN,NaN,NaN
3,"US, DC, Washington",Sales,NaN,f,t,f,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales
4,"US, FL, Fort Worth",NaN,NaN,f,t,t,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider


In [6]:
meta_df.isnull().sum()

,0
location,346
department,11547
salary_range,15012
telecommuting,0
has_company_logo,0
has_questions,0
employment_type,3471
required_experience,7050
required_education,8105
industry,4903


In [7]:
meta_df = meta_df.drop(columns=['salary_range', 'department'])
meta_df.head()

,location,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function
0,"US, NY, New York",f,t,f,Other,Internship,NaN,NaN,Marketing
1,"NZ, , Auckland",f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service
2,"US, IA, Wever",f,t,f,NaN,NaN,NaN,NaN,NaN
3,"US, DC, Washington",f,t,f,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales
4,"US, FL, Fort Worth",f,t,t,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider


In [8]:
meta_df = meta_df.fillna("Unknown")
meta_df.isnull().sum()

,0
location,0
telecommuting,0
has_company_logo,0
has_questions,0
employment_type,0
required_experience,0
required_education,0
industry,0
function,0


In [9]:
# Convert t/f to 1/0
meta_df['telecommuting'] = meta_df['telecommuting'].map({'t': 1, 'f': 0})
meta_df['has_company_logo'] = meta_df['has_company_logo'].map({'t': 1, 'f': 0})
meta_df['has_questions'] = meta_df['has_questions'].map({'t': 1, 'f': 0})

meta_df.head()

,location,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function
0,"US, NY, New York",0,1,0,Other,Internship,Unknown,Unknown,Marketing
1,"NZ, , Auckland",0,1,0,Full-time,Not Applicable,Unknown,Marketing and Advertising,Customer Service
2,"US, IA, Wever",0,1,0,Unknown,Unknown,Unknown,Unknown,Unknown
3,"US, DC, Washington",0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales
4,"US, FL, Fort Worth",0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider


In [10]:
meta_encoded = pd.get_dummies(meta_df, drop_first=True)

meta_encoded.shape

(17880, 3301)

In [11]:
from sklearn.ensemble import IsolationForest

iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.05,   # assume 5% anomalies
    random_state=42
)

iso_model.fit(meta_encoded)

IsolationForest(contamination=0.05, random_state=42)

In [14]:
# Create clean feature matrix again (remove anomaly columns if present)
feature_cols = [col for col in meta_encoded.columns
                if col not in ['anomaly_score', 'anomaly_label']]

X = meta_encoded[feature_cols]

# Now predict
anomaly_scores = iso_model.decision_function(X)
anomaly_labels = iso_model.predict(X)

# Attach results
meta_encoded['anomaly_score'] = anomaly_scores
meta_encoded['anomaly_label'] = anomaly_labels

meta_encoded['anomaly_label'].value_counts()

,count
anomaly_label,
1,16986
-1,894


In [15]:
# Attach true label
meta_encoded['fraudulent'] = df['fraudulent'].values

# Crosstab
import pandas as pd

pd.crosstab(meta_encoded['anomaly_label'], meta_encoded['fraudulent'])

fraudulent,f,t
anomaly_label,,
-1,821,73
1,16193,793


Isolation Forest on structured metadata alone provides limited fraud detection capability (~8% recall), indicating that textual features likely contain stronger fraud signals.

In [16]:
outlier_results = meta_encoded[['anomaly_score', 'anomaly_label']]
outlier_results.to_csv("outlier_stream_results.csv", index=False)

In [18]:
from google.colab import files
files.download("outlier_stream_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>